# Phase 4 — Submission 04: Model 3 Statutory Precision Full-MHA Decoder (Houlsby Bottleneck Adapters)

**Task:** Court Judgment Prediction (CJPE) in Indian Legal Documents (ILDC Benchmark)  
**Task Type:** Document-Level Binary Classification (`0` = Dismissed/Rejected, `1` = Allowed/Accepted)  
**Primary Evaluation Metric:** Macro-F1 Score  
**Target Backbone:** Phase 1 Model 3 Statutory Precision Full-MHA Decoder (~109.5M parameters, $L=12, d=768, H=12$, SwiGLU $d_{ff}=2048$, RoPE)  
**Pretrained Hub Location:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `model3-statutory-mha-110m`)  
**Target Hub Destination:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `Phase_4/sub04_model3_houlsby/`)  

---

### Strategic Architectural Role & Residual Bottleneck Adapters
This notebook implements **Houlsby Residual Bottleneck Adapters** (Houlsby et al., ICML 2019) on Model 3 Statutory Precision:
1. **Adapter Family Representation:** Fulfills the requirement to evaluate residual bottleneck module insertion on Full Multi-Head Attention (12 heads, no grouped query sharing).
2. **Dual Bottleneck Injection per Block:** Injects 2 residual bottleneck MLPs ($d_{\text{bottleneck}} = 56$, SiLU non-linearity) per transformer layer:
   - One bottleneck module after Multi-Head Attention (`self_attn.o_proj`).
   - One bottleneck module after Feed-Forward SwiGLU (`mlp.down_proj`).
3. **Parameter Footprint:** The entire 109.5M base backbone is 100% frozen. Only the bottleneck adapter layers (~2.08M params, $\approx 1.87\%$) plus chunk attention pooling and classification head are trainable.
4. **Exact Identity Initialization:** The up-projection layer is initialized to exact zeros ($W_{\text{up}} = 0, b_{\text{up}} = 0$), ensuring that at step 0 the adapted model computes the exact forward pass of the pretrained base model: $y_{\text{adapted}} = y + 0 = y$.
5. **Tail-Heavy Multi-Chunk Attention Pooling:** Documents are segmented into non-overlapping 1024-token chunks. The last $K=4$ chunks are pooled with learned attention over $[\text{MeanPool}(\mathbf{H}) \,;\, \mathbf{h}_{\text{last}}] \in \mathbb{R}^{1536}$.
6. **Self-Preservation & Hub Synchronization:** Preserves and stages the executed notebook itself into `checkpoint_sub04/` and atomically pushes all artifacts (`houlsby_adapter_weights.pt`, `head.pt`, `attn_pool.pt`, `sub04_test_probs.npy`, `dev_probs.npy`, `val_metrics.json`, `loss_plot.png`, `submission.zip`, and `sub04_model3_houlsby.ipynb`) to the Hugging Face Hub.


In [ ]:
# Cell 1: Imports, Environment Setup & Upfront Hugging Face Authentication
import os
import sys

# 0. Google Colab & Environment Auto-Configuration
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Detected Google Colab environment! Installing required dependencies (peft, transformers, safetensors, accelerate)...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.38.0", "peft>=0.9.0", "safetensors", "huggingface_hub", "accelerate"])
    print("Google Colab dependencies successfully installed!")
import json
import math
import time
import shutil
import zipfile
import random
import getpass
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import PreTrainedTokenizerFast, AutoConfig, AutoModel, get_cosine_schedule_with_warmup
from safetensors.torch import load_file as safetensors_load_file
from huggingface_hub import hf_hub_download, login, get_token, HfApi

# 1. Reproducible Seed Initialization
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Compute Device Configuration with Architecture Compatibility Guard
def get_safe_device(preferred_device: Optional[str] = None) -> torch.device:
    if preferred_device:
        return torch.device(preferred_device)
    if torch.cuda.is_available():
        try:
            t = torch.zeros(1, device="cuda")
            _ = t + 1
            return torch.device("cuda")
        except Exception as e:
            print(f"Notice: CUDA is present but incompatible with current PyTorch build ({e}). Falling back to CPU.")
            return torch.device("cpu")
    return torch.device("cpu")

device = get_safe_device()
print(f"Active Compute Device: {device}")
if device.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

# 3. Repository and Target Configuration
MODEL_REPO = "sarweshbarnwal/CS781-Legal-Models"
TARGET_SUBFOLDER = "Phase_4/sub04_model3_houlsby"
DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
BACKBONE_SUBFOLDER = "model3-statutory-mha-110m"
OUTPUT_DIR = "checkpoint_sub04"

# 4. Upfront Hugging Face Authentication
def resolve_hf_token(token: Optional[str] = None) -> Optional[str]:
    if token and str(token).strip():
        resolved = str(token).strip()
        try:
            login(token=resolved, add_to_git_credential=False)
        except Exception:
            pass
        return resolved

    for env_var in ["HF_TOKEN", "HF", "HUGGINGFACE_TOKEN"]:
        val = os.environ.get(env_var)
        if val and val.strip():
            resolved = val.strip()
            try:
                login(token=resolved, add_to_git_credential=False)
            except Exception:
                pass
            return resolved

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for sec_name in ["HF_TOKEN", "HF", "HUGGINGFACE_TOKEN"]:
            try:
                val = user_secrets.get_secret(sec_name)
                if val and val.strip():
                    resolved = val.strip()
                    try:
                        login(token=resolved, add_to_git_credential=False)
                    except Exception:
                        pass
                    return resolved
            except Exception:
                pass
    except Exception:
        pass

    try:
        hub_tok = get_token()
        if hub_tok and hub_tok.strip():
            return hub_tok.strip()
    except Exception:
        pass

    return None

HARDCODED_HF_TOKEN = "YOUR_HF_TOKEN"
HF_TOKEN = resolve_hf_token(HARDCODED_HF_TOKEN) if 'resolve_hf_token' in globals() else HARDCODED_HF_TOKEN
os.environ["HF_TOKEN"] = HF_TOKEN
try:
    login(token=HF_TOKEN, add_to_git_credential=False)
except Exception:
    pass
print(f"Hugging Face Authentication verified with hardcoded key (token prefix: '{HF_TOKEN[:4]}****').")


In [ ]:
# Cell 2: Universal Dataset Loader for Phase 4 (Court Judgment Prediction)
@dataclass
class LegalJudgmentDoc:
    doc_id: str
    text: str
    label: Optional[int] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        if self.label is not None:
            self.label = int(self.label)
            if self.label not in (0, 1):
                raise ValueError(f"Document {self.doc_id} has invalid binary label: {self.label}")

def _normalize_dataset_records(raw_data: Any, split_name: str) -> List[LegalJudgmentDoc]:
    docs: List[LegalJudgmentDoc] = []
    if isinstance(raw_data, dict):
        for k, v in raw_data.items():
            if isinstance(v, dict):
                text = v.get("text") or v.get("document") or ""
                lbl = v.get("label") if "label" in v else v.get("labels", v.get("pred", None))
                docs.append(LegalJudgmentDoc(doc_id=str(k), text=str(text), label=lbl, metadata=v))
            elif isinstance(v, str):
                docs.append(LegalJudgmentDoc(doc_id=str(k), text=v, label=None))
    elif isinstance(raw_data, list):
        for idx, item in enumerate(raw_data):
            if isinstance(item, dict):
                doc_id = str(item.get("id") or item.get("doc_id") or f"{split_name}_{idx}")
                text = str(item.get("text") or item.get("document") or "")
                lbl = item.get("label") if "label" in item else item.get("labels", item.get("pred", None))
                docs.append(LegalJudgmentDoc(doc_id=doc_id, text=text, label=lbl, metadata=item))
            else:
                raise ValueError(f"Unrecognized item format at index {idx} in {split_name}")
    else:
        raise ValueError(f"Unrecognized dataset container type: {type(raw_data)} in {split_name}")

    if split_name in ("train", "dev"):
        for doc in docs:
            if doc.label is None:
                raise ValueError(f"FATAL: Missing label in {split_name} document: '{doc.doc_id}'")

    return docs

def load_phase4_splits(repo_id: str = DATASET_REPO, token: Optional[str] = None):
    splits_files = {
        "train": "Phase_4_CJPE/Train/train.json",
        "dev": "Phase_4_CJPE/Train/dev.json",
        "test": "Phase_4_CJPE/Test/test.json",
    }
    loaded = {}
    for key, path in splits_files.items():
        print(f"Downloading Phase 4 {key} split from '{repo_id}/{path}'...")
        local_path = hf_hub_download(repo_id=repo_id, filename=path, repo_type="dataset", token=token)
        with open(local_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        docs = _normalize_dataset_records(raw_data, key)
        loaded[key] = docs
        print(f"  -> Loaded {len(docs):,} {key} documents.")

    train_docs, dev_docs, test_docs = loaded["train"], loaded["dev"], loaded["test"]
    return train_docs, dev_docs, test_docs

train_docs, dev_docs, test_docs = load_phase4_splits(token=HF_TOKEN)
print(f"Dataset Loaded: {len(train_docs):,} Train, {len(dev_docs):,} Dev, {len(test_docs):,} Test")


In [ ]:
# Cell 3: Model 3 Tokenizer & Tail-Chunking Dataset Pipeline
def load_model3_tokenizer(repo_id: str = MODEL_REPO, token: Optional[str] = None) -> PreTrainedTokenizerFast:
    tok_path = hf_hub_download(repo_id=repo_id, filename=f"{BACKBONE_SUBFOLDER}/tokenizer.json", token=token)
    tok = PreTrainedTokenizerFast(
        tokenizer_file=tok_path,
        bos_token="<bos>",
        eos_token="<eos>",
        pad_token="<pad>",
        unk_token="<unk>"
    )
    tok.padding_side = "left"
    assert tok.pad_token_id == 3 and tok.eos_token_id == 1, f"Special token ID mismatch: pad={tok.pad_token_id}, eos={tok.eos_token_id}"
    return tok

tokenizer = load_model3_tokenizer(token=HF_TOKEN)

@dataclass
class PreprocessedDocumentBatch:
    doc_id: str
    num_chunks: int
    input_ids: torch.Tensor       # Shape: (M, 1024)
    attention_mask: torch.Tensor  # Shape: (M, 1024)
    label: Optional[int] = None

def preprocess_doc_tail_chunks(
    doc: LegalJudgmentDoc,
    tokenizer: PreTrainedTokenizerFast,
    chunk_size: int = 1024,
    k_tail: int = 4
) -> PreprocessedDocumentBatch:
    tokens = tokenizer(doc.text, add_special_tokens=False, return_attention_mask=False)["input_ids"]
    
    if len(tokens) == 0:
        tokens = [tokenizer.eos_token_id]

    chunks = [tokens[i : i + chunk_size] for i in range(0, len(tokens), chunk_size)]
    tail_chunks = chunks[-k_tail:]
    M = len(tail_chunks)

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 3
    input_ids_list = []
    attention_mask_list = []

    for c in tail_chunks:
        pad_len = chunk_size - len(c)
        padded_ids = [pad_id] * pad_len + c
        mask = [0] * pad_len + [1] * len(c)
        input_ids_list.append(padded_ids)
        attention_mask_list.append(mask)

    return PreprocessedDocumentBatch(
        doc_id=doc.doc_id,
        num_chunks=M,
        input_ids=torch.tensor(input_ids_list, dtype=torch.long),
        attention_mask=torch.tensor(attention_mask_list, dtype=torch.long),
        label=doc.label
    )

class CJPEBatchDataset(Dataset):
    def __init__(self, docs: List[LegalJudgmentDoc], tokenizer: PreTrainedTokenizerFast, k_tail: int = 4):
        self.docs = docs
        self.tokenizer = tokenizer
        self.k_tail = k_tail

    def __len__(self) -> int:
        return len(self.docs)

    def __getitem__(self, idx: int) -> PreprocessedDocumentBatch:
        return preprocess_doc_tail_chunks(self.docs[idx], self.tokenizer, chunk_size=1024, k_tail=self.k_tail)

print("Constructing preprocessed dataset instances for Phase 4...")
train_dataset = CJPEBatchDataset(train_docs, tokenizer, k_tail=4)
dev_dataset = CJPEBatchDataset(dev_docs, tokenizer, k_tail=4)
test_dataset = CJPEBatchDataset(test_docs, tokenizer, k_tail=4)

print(f"Datasets Initialized -> Train: {len(train_dataset):,}, Dev: {len(dev_dataset):,}, Test: {len(test_dataset):,}")


In [ ]:
# Cell 4: Upfront Dataloaders & Optimization Step Calibration
BATCH_SIZE = 1           # 1 document per micro-batch (processes up to K=4 chunks of 1024 tokens = 4096 tokens)
GRAD_ACCUM_STEPS = 8   # Effective batch size = 8 documents     # Effective batch size = 4 documents
EPOCHS = 1   # Calibrated for 1-epoch fast execution (~1.5 hrs)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: b[0],
    drop_last=False
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: b[0]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: b[0]
)

total_optimization_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(0.10 * total_optimization_steps)

print("=" * 65)
print(f"Step Calibration for Houlsby Bottleneck Adapter Training:")
print(f"  Training Documents:       {len(train_loader):,}")
print(f"  Micro-Batch Size:         {BATCH_SIZE} document (up to 4 x 1024 = 4096 tokens)")
print(f"  Gradient Accumulation:    {GRAD_ACCUM_STEPS} steps")
print(f"  Total Epochs:             {EPOCHS}")
print(f"  Total Optimization Steps: {total_optimization_steps:,}")
print(f"  Warmup Steps (10%):       {warmup_steps:,}")
print("=" * 65)


In [ ]:
# Cell 5: Model 3 Backbone Loading from Hugging Face Hub
# Model 3 Statutory Precision Full-MHA Decoder (~109.5M parameters, L=12, d=768, H=12, SwiGLU d_ff=2048)

print(f"Downloading Model 3 configuration and weights from '{MODEL_REPO}/{BACKBONE_SUBFOLDER}'...")
cfg_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/config.json", token=HF_TOKEN)
weights_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/model.safetensors", token=HF_TOKEN)

config = AutoConfig.from_pretrained(cfg_path)
config.use_cache = False
config._name_or_path = MODEL_REPO
base_model = AutoModel.from_config(config)
base_model.config._name_or_path = MODEL_REPO

raw_state_dict = safetensors_load_file(weights_path)
cleaned_sd = {}
for k, v in raw_state_dict.items():
    if k.startswith("model."):
        cleaned_sd[k[len("model."):]] = v
    elif not k.startswith("lm_head."):
        cleaned_sd[k] = v

load_res = base_model.load_state_dict(cleaned_sd, strict=True)
assert len(load_res.missing_keys) == 0, f"Missing keys when loading backbone: {load_res.missing_keys}"
assert len(load_res.unexpected_keys) == 0, f"Unexpected keys when loading backbone: {load_res.unexpected_keys}"
print(f"Base model loaded successfully with strict=True! 100% of Phase 1 weights matched ({len(cleaned_sd)} tensors).")
print(f"Base Model 3 Statutory Precision Full-MHA Decoder loaded successfully! (L={config.num_hidden_layers}, d={config.hidden_size})")


In [ ]:
# Cell 6: Houlsby Residual Bottleneck Adapter Architecture & Injection
class ResidualBottleneckAdapter(nn.Module):
    """
    Houlsby / Pfeiffer style 2-layer residual bottleneck MLP adapter:
      h_adapted = h + W_up(Dropout(SiLU(W_down(h))))
    W_up is initialized to exact zero so the adapter acts as an identity at step 0.
    """
    def __init__(self, d_model: int = 768, bottleneck_dim: int = 56, dropout: float = 0.05):
        super().__init__()
        self.down_proj = nn.Linear(d_model, bottleneck_dim)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.up_proj = nn.Linear(bottleneck_dim, d_model)

        # Initialize down projection with Kaiming uniform and up projection with exact zeros
        nn.init.kaiming_uniform_(self.down_proj.weight, a=math.sqrt(5))
        nn.init.zeros_(self.down_proj.bias)
        nn.init.zeros_(self.up_proj.weight)
        nn.init.zeros_(self.up_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        down = self.act(self.down_proj(x))
        up = self.up_proj(self.dropout(down))
        return residual + up

class HoulsbyLinearWrapper(nn.Module):
    """
    Wraps a frozen projection linear layer (o_proj or down_proj) with a trailing ResidualBottleneckAdapter:
      y = adapter(frozen_linear(x))
    """
    def __init__(self, base_linear: nn.Linear, d_model: int = 768, bottleneck_dim: int = 56, dropout: float = 0.05):
        super().__init__()
        self.base_linear = base_linear
        # Ensure base projection weights are strictly frozen
        for p in self.base_linear.parameters():
            p.requires_grad = False
        self.adapter = ResidualBottleneckAdapter(d_model=d_model, bottleneck_dim=bottleneck_dim, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.base_linear(x)
        return self.adapter(y)

# 1. Freeze 100% of Base Model Parameters
for param in base_model.parameters():
    param.requires_grad = False

# 2. Inject Houlsby Bottleneck Adapters after Self-Attention and Feed-Forward Blocks across all 12 layers
BOTTLENECK_DIM = 56
adapter_count = 0

for idx, layer in enumerate(base_model.layers):
    # Adapter after Self-Attention (post-o_proj)
    layer.self_attn.o_proj = HoulsbyLinearWrapper(
        layer.self_attn.o_proj,
        d_model=config.hidden_size,
        bottleneck_dim=BOTTLENECK_DIM,
        dropout=0.05
    )
    # Adapter after Feed-Forward SwiGLU (post-down_proj)
    layer.mlp.down_proj = HoulsbyLinearWrapper(
        layer.mlp.down_proj,
        d_model=config.hidden_size,
        bottleneck_dim=BOTTLENECK_DIM,
        dropout=0.05
    )
    adapter_count += 2

model = base_model.to(device)

total_backbone = sum(p.numel() for p in model.parameters())
trainable_adapters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("")
print("=" * 65)
print(f"Model 3 Statutory Precision Full-MHA Backbone Loaded:")
print(f"  Total Model 3 Parameters:          {total_backbone:,}")
print(f"  Houlsby Adapters Injected:         {adapter_count} modules ({adapter_count // 2} layers x 2 adapters)")
print(f"  Adapter Bottleneck Dimension:      {BOTTLENECK_DIM} (SiLU non-linearity)")
print(f"  Trainable Adapter Parameters:      {trainable_adapters:,} ({trainable_adapters / total_backbone * 100:.2f}%)")
print("=" * 65)
print("")

# Multi-GPU DataParallel Configuration
is_multi_gpu = torch.cuda.is_available() and torch.cuda.device_count() > 1
if is_multi_gpu:
    num_gpus = torch.cuda.device_count()
    print(f"Detected {num_gpus} GPUs! Enabling torch.nn.DataParallel on Houlsby model across all {num_gpus} GPUs...")
    parallel_model = nn.DataParallel(model)
else:
    parallel_model = model


In [ ]:
# Cell 7: Chunk Concat Representation & Learned Attention Pooling
def extract_chunk_concat_representations(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    Computes Concat(MaskedMeanPool, LastTokenHiddenState) for each chunk in the document:
      h_chunk = [MeanPool(H_valid) ; h_last_token] in R^(2 * d_model)
    For Model 3 (d_model=768): h_chunk in R^(1536)
    """
    M, seq_len, d_model = hidden_states.shape
    chunk_vectors: List[torch.Tensor] = []

    for m in range(M):
        valid_mask = attention_mask[m].unsqueeze(-1).to(hidden_states.dtype)  # (seq_len, 1)
        valid_tokens = hidden_states[m] * valid_mask                         # (seq_len, d_model)
        
        token_count = valid_mask.sum().clamp(min=1.0)
        mean_pool = valid_tokens.sum(dim=0) / token_count                   # (d_model,)
        last_token = hidden_states[m, -1, :]                                # (d_model,)

        chunk_vectors.append(torch.cat([mean_pool, last_token], dim=-1))    # (2 * d_model,)

    return torch.stack(chunk_vectors, dim=0)                                # (M, 2 * d_model)

class ChunkConcatAttentionPooling(nn.Module):
    """
    Learned scalar attention pooling over M chunk representations:
      u_m = tanh(W_a h_m + b_a)
      alpha_m = softmax(v_a^T u_m)
      h_doc = sum(alpha_m * h_m) in R^(2 * d_model)
    """
    def __init__(self, d_model: int = 768, d_attn: int = 128):
        super().__init__()
        self.in_dim = d_model * 2  # 768 * 2 = 1536
        self.W_a = nn.Linear(self.in_dim, d_attn)
        self.v_a = nn.Linear(d_attn, 1, bias=False)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_a.weight)
        nn.init.zeros_(self.W_a.bias)
        nn.init.xavier_uniform_(self.v_a.weight)

    def forward(self, chunk_embeds: torch.Tensor) -> torch.Tensor:
        u = torch.tanh(self.W_a(chunk_embeds))           # (M, d_attn)
        scores = self.v_a(u).squeeze(-1)                 # (M,)
        weights = F.softmax(scores, dim=0)               # (M,)
        doc_embed = torch.sum(weights.unsqueeze(-1) * chunk_embeds, dim=0) # (2 * d_model,)
        return doc_embed

class CourtJudgmentHead(nn.Module):
    """
    2-Layer MLP Classification Head for Court Judgment Prediction:
      LayerNorm(2 * d_model) -> Linear(2 * d_model, hidden_dim) -> GELU -> Dropout(0.1) -> Linear(hidden_dim, 1)
    """
    def __init__(self, d_model: int = 768, hidden_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.in_dim = d_model * 2  # 1536
        self.layer_norm = nn.LayerNorm(self.in_dim)
        self.fc1 = nn.Linear(self.in_dim, hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, doc_embed: torch.Tensor) -> torch.Tensor:
        if doc_embed.dim() == 1:
            doc_embed = doc_embed.unsqueeze(0)
        h = self.layer_norm(doc_embed)
        h = self.dropout(self.act(self.fc1(h)))
        return self.fc2(h)

attn_pool = ChunkConcatAttentionPooling(d_model=768, d_attn=128).to(device)
head = CourtJudgmentHead(d_model=768, hidden_dim=256, dropout=0.1).to(device)

trainable_pool = sum(p.numel() for p in attn_pool.parameters() if p.requires_grad)
trainable_head = sum(p.numel() for p in head.parameters() if p.requires_grad)
total_trainable = trainable_adapters + trainable_pool + trainable_head

print(f"Initialized ChunkConcatAttentionPooling ({trainable_pool:,} params) and CourtJudgmentHead ({trainable_head:,} params).")
print(f"Total Trainable System Parameters: {total_trainable:,} ({total_trainable / total_backbone * 100:.2f}% of backbone)")


In [ ]:
# Cell 8: Class Balance Analysis & BCEWithLogitsLoss Objective
train_labels = [int(d.label) for d in train_docs if d.label is not None]
pos_count = sum(train_labels)
neg_count = len(train_labels) - pos_count
imbalance_ratio = neg_count / max(1, pos_count)

pos_weight = torch.tensor([imbalance_ratio], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Train Set Class Balance: {pos_count} Positive (Allowed), {neg_count} Negative (Dismissed)")
print(f"Imbalance Ratio: {imbalance_ratio:.4f} -> BCEWithLogitsLoss pos_weight: {pos_weight.item():.4f}")


In [ ]:
# Cell 9: Differential Optimization Setup & Cosine Schedule
LR_BACKBONE = 4.0e-4   # Bottleneck adapter learning rate
LR_HEAD = 1.0e-3       # Pooling and Classification Head learning rate

param_groups = [
    {
        "params": [p for p in model.parameters() if p.requires_grad],
        "lr": LR_BACKBONE,
        "weight_decay": 0.01,
    },
    {
        "params": list(attn_pool.parameters()) + list(head.parameters()),
        "lr": LR_HEAD,
        "weight_decay": 0.01,
    },
]

optimizer = torch.optim.AdamW(param_groups)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimization_steps
)

print(f"Optimizer: AdamW | Houlsby Adapter LR: {LR_BACKBONE} | Head/Pool LR: {LR_HEAD} | Steps: {total_optimization_steps} (Warmup: {warmup_steps})")


In [ ]:
# Cell 10: Evaluation Function & Optimal Decision Threshold Sweeper
def compute_binary_macro_f1(y_true: List[int], y_pred: List[int]) -> float:
    tp = {0: 0, 1: 0}
    fp = {0: 0, 1: 0}
    fn = {0: 0, 1: 0}
    for gt, pr in zip(y_true, y_pred):
        gt, pr = int(gt), int(pr)
        if gt == pr:
            tp[gt] += 1
        else:
            fp[pr] += 1
            fn[gt] += 1
    f1s = []
    for c in [0, 1]:
        p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
        r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        f1s.append(f1)
    return sum(f1s) / 2.0

def sweep_decision_threshold(y_true: List[int], y_probs: List[float]) -> Tuple[float, float, Dict[float, float]]:
    best_tau = 0.50
    best_f1 = -1.0
    results: Dict[float, float] = {}
    for tau in np.arange(0.10, 0.91, 0.01):
        tau = round(float(tau), 4)
        preds = [1 if p >= tau else 0 for p in y_probs]
        f1 = compute_binary_macro_f1(y_true, preds)
        results[tau] = f1
        if f1 > best_f1:
            best_f1 = f1
            best_tau = tau
    return best_tau, best_f1, results

def evaluate_dev(model, attn_pool, head, dev_dataset, criterion, device, parallel_model: Optional[nn.Module] = None) -> Tuple[float, float, float, List[int], List[float]]:
    model.eval()
    attn_pool.eval()
    head.eval()
    total_loss = 0.0
    all_golds: List[int] = []
    all_probs: List[float] = []

    with torch.no_grad():
        for doc_batch in dev_dataset:
            input_ids = doc_batch.input_ids.to(device)
            attention_mask = doc_batch.attention_mask.to(device)
            target = torch.tensor([doc_batch.label], dtype=torch.float32, device=device)

            use_dp = parallel_model is not None and torch.cuda.device_count() > 1
            if use_dp and input_ids.shape[0] >= torch.cuda.device_count():
                outputs = parallel_model(input_ids=input_ids, attention_mask=attention_mask)
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
            doc_embed = attn_pool(chunk_reprs)
            logit = head(doc_embed).squeeze(-1)

            loss = criterion(logit, target)
            total_loss += loss.item()

            prob = torch.sigmoid(logit).item()
            all_probs.append(prob)
            all_golds.append(int(doc_batch.label))

    avg_loss = total_loss / max(1, len(all_golds))
    best_tau, best_macro_f1, _ = sweep_decision_threshold(all_golds, all_probs)
    return avg_loss, best_macro_f1, best_tau, all_golds, all_probs


In [ ]:
# Cell 11: 1-Epoch Houlsby Residual Bottleneck Adapter Training Loop with Auto-Resume & Periodic HF Sync
output_dir = OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)

best_dev_f1 = 0.0
best_tau = 0.50
history = {"train_loss": [], "dev_loss": [], "dev_macro_f1": []}

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def stage_and_push_checkpoint(step, tag="periodic"):
    global best_dev_f1, best_tau, global_step
    ckpt_path = os.path.join(output_dir, "checkpoint_resume.pt")
    adapter_weights = {k: v.cpu().clone() for k, v in model.state_dict().items() if "adapter." in k}
    save_dict = {
        "step": step,
        "global_step": global_step,
        "best_dev_f1": best_dev_f1,
        "best_tau": best_tau,
        "adapter_weights": adapter_weights,
        "attn_pool": attn_pool.state_dict(),
        "head": head.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() if use_amp else None,
    }
    torch.save(save_dict, ckpt_path)
    torch.save(adapter_weights, os.path.join(output_dir, "houlsby_adapter_weights.pt"))
    torch.save(attn_pool.state_dict(), os.path.join(output_dir, "attn_pool.pt"))
    torch.save(head.state_dict(), os.path.join(output_dir, "head.pt"))
    print(f"  [Checkpoint Staged] Saved checkpoint at Step {step:,} to '{output_dir}'.")
    try:
        api = HfApi(token=HF_TOKEN)
        api.upload_folder(
            folder_path=output_dir,
            repo_id=MODEL_REPO,
            path_in_repo=TARGET_SUBFOLDER,
            commit_message=f"Sync Checkpoint [{tag}] Step {step} - Best Dev F1: {best_dev_f1:.4f}",
            ignore_patterns=["*.tmp", "__pycache__"]
        )
        print(f"  [Hub Sync] Checkpoint successfully pushed to Hugging Face Hub ({TARGET_SUBFOLDER})!")
    except Exception as e:
        print(f"  [Hub Sync Notice] Background push skipped/failed ({e}). Training continuing uninterrupted.")

resume_path = os.path.join(output_dir, "checkpoint_resume.pt")
if not os.path.exists(resume_path):
    try:
        print(f"Checking Hugging Face Hub for existing resume checkpoint in '{MODEL_REPO}/{TARGET_SUBFOLDER}'...")
        down = hf_hub_download(
            repo_id=MODEL_REPO,
            filename=f"{TARGET_SUBFOLDER}/checkpoint_resume.pt",
            token=HF_TOKEN,
            local_dir=output_dir
        )
        if os.path.exists(down):
            resume_path = down
    except Exception:
        pass

start_step = 0
global_step = 0
if os.path.exists(resume_path):
    try:
        ckpt = torch.load(resume_path, map_location=device)
        start_step = ckpt.get("step", 0)
        global_step = ckpt.get("global_step", 0)
        best_dev_f1 = ckpt.get("best_dev_f1", 0.0)
        best_tau = ckpt.get("best_tau", 0.50)
        if "adapter_weights" in ckpt:
            model.load_state_dict(ckpt["adapter_weights"], strict=False)
        attn_pool.load_state_dict(ckpt["attn_pool"])
        head.load_state_dict(ckpt["head"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if use_amp and ckpt.get("scaler"):
            scaler.load_state_dict(ckpt["scaler"])
        print(f"[SUCCESS] RESUME SUCCESSFUL! Picking up training from Step {start_step:,}/{len(train_dataset):,} (Global Step: {global_step:,}, Best Dev F1: {best_dev_f1:.4f})")
    except Exception as e:
        print(f"Notice: Could not load resume checkpoint ({e}). Starting clean training.")
        start_step = 0

print(f"Commencing {EPOCHS}-epoch Houlsby Bottleneck Adapter Training Loop for Sub 04 (Starting from step {start_step:,})...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    attn_pool.train()
    head.train()
    running_loss = 0.0
    optimizer.zero_grad()

    indices = list(range(len(train_dataset)))
    perm_rng = random.Random(42 + epoch)
    perm_rng.shuffle(indices)

    for step, idx in enumerate(indices):
        if step < start_step:
            continue

        doc_batch = train_dataset[idx]
        input_ids = doc_batch.input_ids.to(device)
        attention_mask = doc_batch.attention_mask.to(device)
        target = torch.tensor([doc_batch.label], dtype=torch.float32, device=device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            if is_multi_gpu and input_ids.shape[0] >= torch.cuda.device_count():
                outputs = parallel_model(input_ids=input_ids, attention_mask=attention_mask)
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
            doc_embed = attn_pool(chunk_reprs)
            logit = head(doc_embed).squeeze(-1)

            loss = criterion(logit, target)
            loss_scaled = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_scaled).backward()
        running_loss += loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(indices):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            torch.nn.utils.clip_grad_norm_(attn_pool.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

        if (step + 1) % (100 * GRAD_ACCUM_STEPS) == 0:
            avg_loss_step = running_loss / (step + 1)
            print(f"  [Epoch {epoch:02d} | Step {step+1:04d}/{len(indices):04d}] Current Loss: {avg_loss_step:.4f} | Global Steps: {global_step}")

        if (step + 1) % 5000 == 0:
            stage_and_push_checkpoint(step + 1, tag=f"step_{step+1}")

        if (step + 1) == len(indices) // 2:
            print("  --- Performing Mid-Epoch Validation Check ---")
            mid_loss, mid_f1, mid_tau, _, _ = evaluate_dev(model, attn_pool, head, dev_dataset, criterion, device, parallel_model=parallel_model)
            print(f"  Mid-Epoch Dev Loss: {mid_loss:.4f} | Dev Macro-F1: {mid_f1:.4f} (tau*: {mid_tau:.2f})")
            if mid_f1 > best_dev_f1:
                best_dev_f1 = mid_f1
                best_tau = mid_tau
                stage_and_push_checkpoint(step + 1, tag=f"mid_epoch_best_{best_dev_f1:.4f}")

    epoch_train_loss = running_loss / len(indices)
    dev_loss, dev_f1, tau_star, _, _ = evaluate_dev(model, attn_pool, head, dev_dataset, criterion, device, parallel_model=parallel_model)

    history["train_loss"].append(epoch_train_loss)
    history["dev_loss"].append(dev_loss)
    history["dev_macro_f1"].append(dev_f1)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} Complete | Train Loss: {epoch_train_loss:.4f} | Dev Loss: {dev_loss:.4f} | Dev Macro-F1: {dev_f1:.4f} (tau*: {tau_star:.2f}) | Global Steps: {global_step}")

    if dev_f1 > best_dev_f1 or best_dev_f1 == 0.0:
        best_dev_f1 = dev_f1
        best_tau = tau_star
        stage_and_push_checkpoint(len(indices), tag=f"final_epoch_best_{best_dev_f1:.4f}")
        print(f"  -> NEW BEST Dev Macro-F1: {best_dev_f1:.4f} (tau*: {best_tau:.2f})! Model checkpoint saved and pushed.")

total_train_time = time.time() - start_time
print(f"\nTraining completed in {total_train_time / 60:.2f} minutes. Best Dev Macro-F1: {best_dev_f1:.4f} at tau*={best_tau:.2f}")


In [ ]:
# Cell 12: Test Inference, Calibrated Threshold Prediction & Codabench Packaging
# 1. Load Best Saved Checkpoint Weights
adapter_path = os.path.join(output_dir, "houlsby_adapter_weights.pt")
pool_path = os.path.join(output_dir, "attn_pool.pt")
head_path = os.path.join(output_dir, "head.pt")

if os.path.exists(adapter_path):
    print("Loading best Houlsby adapter and head weights for final evaluation...")
    saved_weights = torch.load(adapter_path, map_location=device)
    model.load_state_dict(saved_weights, strict=False)
    attn_pool.load_state_dict(torch.load(pool_path, map_location=device))
    head.load_state_dict(torch.load(head_path, map_location=device))

# 2. Final Evaluation on Dev Split to confirm optimal tau*
print("Executing final dev evaluation sweep...")
final_dev_loss, final_dev_f1, optimal_tau, dev_golds, dev_probs = evaluate_dev(
    model, attn_pool, head, dev_dataset, criterion, device, parallel_model=parallel_model
)
print(f"Final Dev Macro-F1: {final_dev_f1:.4f} at Optimal Decision Threshold tau* = {optimal_tau:.2f}")

np.save(os.path.join(output_dir, "dev_probs.npy"), np.array(dev_probs, dtype=np.float32))

# 3. Predict on Test Split
model.eval()
attn_pool.eval()
head.eval()
test_probs: List[float] = []

print(f"Generating test predictions across {len(test_dataset):,} test documents...")
with torch.no_grad():
    for doc_batch in test_dataset:
        input_ids = doc_batch.input_ids.to(device)
        attention_mask = doc_batch.attention_mask.to(device)

        if is_multi_gpu and input_ids.shape[0] >= torch.cuda.device_count():
            outputs = parallel_model(input_ids=input_ids, attention_mask=attention_mask)
        else:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
        doc_embed = attn_pool(chunk_reprs)
        logit = head(doc_embed).squeeze(-1)

        prob = torch.sigmoid(logit).item()
        test_probs.append(prob)

sub04_test_probs_path = os.path.join(output_dir, "sub04_test_probs.npy")
np.save(sub04_test_probs_path, np.array(test_probs, dtype=np.float32))
print(f"Exported test probabilities to '{sub04_test_probs_path}' ({len(test_probs)} items).")

# 4. Generate Codabench Formatted prediction.json using Optimal Threshold
prediction_list = []
for doc, p in zip(test_docs, test_probs):
    pred_label = 1 if p >= optimal_tau else 0
    prediction_list.append({"id": doc.doc_id, "pred": int(pred_label)})

pred_path = os.path.join(output_dir, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(prediction_list, f, indent=2)

print(f"Exported '{pred_path}' with {len(prediction_list)} predictions (Threshold tau* = {optimal_tau:.2f}).")

# 5. Export Performance Metrics and Evaluation Report
metrics = {
    "submission_id": "sub04_model3_houlsby",
    "model_backbone": "Model 3 Statutory Precision Full-MHA Decoder (109.5M)",
    "adaptation_method": "Houlsby Residual Bottleneck Adapters (d_bottleneck=56, SiLU)",
    "adapter_locations": ["post-self_attn.o_proj", "post-mlp.down_proj"],
    "best_dev_macro_f1": float(final_dev_f1),
    "optimal_decision_threshold": float(optimal_tau),
    "dev_loss": float(final_dev_loss),
    "total_optimization_steps": int(total_optimization_steps),
    "epochs_trained": int(EPOCHS),
}
with open(os.path.join(output_dir, "val_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

report = {
    "metrics": metrics,
    "training_history": history,
}
with open(os.path.join(output_dir, "training_evaluation_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

# 6. Plot Loss Curve & Macro-F1 Progression
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="Train Loss", marker="o")
plt.plot(range(1, len(history["dev_loss"]) + 1), history["dev_loss"], label="Dev Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sub 04: Houlsby Adapter Loss Curve")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history["dev_macro_f1"]) + 1), history["dev_macro_f1"], label="Dev Macro-F1", marker="^", color="green")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title(f"Dev Macro-F1 (Peak: {final_dev_f1:.4f} at tau*={optimal_tau:.2f})")
plt.legend()
plt.grid(True)

plt.tight_layout()
plot_path = os.path.join(output_dir, "loss_plot.png")
plt.savefig(plot_path, dpi=150)
plt.close()
print(f"Saved training visualization to '{plot_path}'.")

# 7. Strict Codabench Packaging: Package prediction.json directly at the zip root
zip_path = os.path.join(output_dir, "submission.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

# Integrity Verification of submission.zip
with zipfile.ZipFile(zip_path, "r") as zf:
    infolist = zf.infolist()
    assert len(infolist) == 1, f"Expected exactly 1 file in submission.zip, found {len(infolist)}"
    assert infolist[0].filename == "prediction.json", f"Found '{infolist[0].filename}' instead of 'prediction.json' at root"

assert len(prediction_list) == len(test_docs), f"Test doc count mismatch: {len(prediction_list)} vs {len(test_docs)}"
for item in prediction_list:
    assert item["pred"] in (0, 1), f"Invalid prediction: {item['pred']}"

print(f"Codabench submission successfully verified and packaged: '{zip_path}' ({os.path.getsize(zip_path):,} bytes, {len(prediction_list)} test cases).")


In [ ]:
# Cell 13: Local Artifact Verification & Atomic Hugging Face Hub Upload (with Executed Notebook Sync)
notebook_filename = "sub04_model3_houlsby.ipynb"

# 1. Save Tokenizer and Model Configuration
tokenizer.save_pretrained(output_dir)
config.save_pretrained(output_dir)

# 2. Stage/Preserve the Executed Notebook File into output_dir
def stage_executed_notebook(nb_name: str, dest_dir: str):
    search_paths = [
        nb_name,
        os.path.join(os.getcwd(), nb_name),
        os.path.join("/content", nb_name),
        os.path.join("/content", os.path.basename(nb_name)),
        os.path.join("/kaggle/working", nb_name),
        os.path.join("/kaggle/working", os.path.basename(nb_name)),
        "/kaggle/working/.virtual_documents/__notebook_source__.ipynb",
        "/kaggle/working/__notebook_source__.ipynb",
    ]
    staged = False
    for p in search_paths:
        if os.path.exists(p) and os.path.isfile(p):
            dest_file = os.path.join(dest_dir, nb_name)
            shutil.copy2(p, dest_file)
            print(f"Successfully staged executed notebook '{p}' -> '{dest_file}' ({os.path.getsize(dest_file):,} bytes).")
            staged = True
            break
    if not staged:
        try:
            import IPython
            IPython.display.display(IPython.display.Javascript('IPython.notebook.save_notebook()'))
            time.sleep(2)
        except Exception:
            pass
        for p in search_paths:
            if os.path.exists(p) and os.path.isfile(p):
                dest_file = os.path.join(dest_dir, nb_name)
                shutil.copy2(p, dest_file)
                print(f"Successfully staged executed notebook '{p}' -> '{dest_file}'.")
                staged = True
                break
    if not staged:
        print(f"Notice: '{nb_name}' not found locally. Ensure notebook is saved before final upload.")

stage_executed_notebook(notebook_filename, output_dir)

# 3. Write Clean YAML README.md for Hugging Face Hub Compliance
readme_path = os.path.join(output_dir, "README.md")
readme_content = f"""---
base_model: {MODEL_REPO}
tags:
- legal
- legal-nlp
- court-judgment-prediction
- ildc
- adapters
- houlsby
---

# Phase 4 Submission 04: Model 3 Statutory Precision Full-MHA Decoder (Houlsby Bottleneck Adapters)
- **Base Model:** `{MODEL_REPO}` (subfolder: `{BACKBONE_SUBFOLDER}`)
- **Task:** Court Judgment Prediction (CJPE - Binary Classification)
- **Role:** Adapter Family Evaluation (residual bottleneck modules on Full MHA)
- **Adaptation Method:** Houlsby Residual Bottleneck Adapters ($d_{{\\text{{bottleneck}}}}=56$, SiLU non-linearity)
- **Adapter Locations:** Injected post-Self-Attention (`o_proj`) and post-Feed-Forward (`down_proj`) across all 12 layers
- **Best Dev Macro-F1:** {final_dev_f1:.4f} (Optimal Decision Threshold tau*: {optimal_tau:.2f})
- **Trainable Parameters:** {total_trainable:,} / {total_backbone:,} ({total_trainable / total_backbone * 100:.2f}%)
- **Test Predictions:** Available in `prediction.json` and packaged in `submission.zip`
"""
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

# 4. Verify All Required Artifacts Exist Locally Before Upload
required_files = [
    "houlsby_adapter_weights.pt",
    "attn_pool.pt",
    "head.pt",
    "config.json",
    "tokenizer.json",
    "val_metrics.json",
    "training_evaluation_report.json",
    "loss_plot.png",
    "dev_probs.npy",
    "sub04_test_probs.npy",
    "prediction.json",
    "submission.zip",
    "README.md"
]

missing_files = [f for f in required_files if not os.path.exists(os.path.join(output_dir, f))]
assert not missing_files, f"Cannot proceed with upload! Missing local files: {missing_files}"
print(f"All required local artifacts verified present in '{output_dir}'!")

# 5. Atomic Single-Commit Folder Upload to Hugging Face Hub
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter Hugging Face Write Token: ")

api = HfApi(token=HF_TOKEN)
print(f"Initiating atomic upload to '{MODEL_REPO}' under '{TARGET_SUBFOLDER}'...")
api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    path_in_repo=TARGET_SUBFOLDER,
    commit_message=f"Upload Phase 4 Sub 04 (Model 3 Houlsby Adapters) - Dev F1: {final_dev_f1:.4f}",
    token=HF_TOKEN
)
print(f"Upload complete! Artifacts & executed notebook published to: https://huggingface.co/{MODEL_REPO}/tree/main/{TARGET_SUBFOLDER}")
